<a href="https://colab.research.google.com/github/cgm2179/indoor-walk-test/blob/main/SIM/phase_c_train_colab_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase C v3 — UNet Path-Loss Surrogate Training
**Why v3**: the spec's 8-channel input stalls at ~16 dB val RMSE — the net has to
infer 20·log₁₀(d) from a σ=2-cell blob. v3 adds an explicit **log-distance channel**
(geometry, not a Tx parameter — R2 untouched) and masks the loss to **indoor cells**
so no capacity is spent on the outside padding. Expect val RMSE to drop under 10 dB
within the first couple of epochs and head toward the ≤3 dB target.

Everything else as v2: dataset auto-discovered anywhere in the top two Drive levels,
checkpoints to Drive every improvement, ONNX copy kept on Drive, baselines
(FSPL / fitted log-distance / 38.901 InH exact-LOS), R5 test discipline,
0.1 dB ONNX parity gate.

**Runtime → GPU.** ~7–8 min/epoch on a T4 at base width 64.

In [9]:
import os
REPO = 'github.com/cgm2179/indoor-walk-test.git'
if not os.path.exists('indoor-walk-test'):
    if os.system(f'git clone --depth 1 https://{REPO}') != 0:
        from getpass import getpass
        token = getpass('GitHub fine-grained token: ')
        assert os.system(f'git clone --depth 1 https://{token}@{REPO}') == 0, 'clone failed'
%cd indoor-walk-test
import torch, json, numpy as np
print('GPU:', torch.cuda.is_available())
manifest = json.load(open('SIM/manifest.json'))
NORM = manifest['norm']; PL_LO, PL_RNG = NORM['pl_min_db'], NORM['pl_range_db']
H, W = manifest['grid_shape']; SIGMA = manifest['tx_blob_sigma_cells']
CELL = manifest['cell_size_m']; DN = manifest.get('dist_channel_norm', 3.0)
IN_CH = 9
print('grid', (H, W), '| clip', PL_LO, '-', PL_LO + PL_RNG, 'dB |', IN_CH, 'channels')

GitHub fine-grained token: ··········
/content/indoor-walk-test/indoor-walk-test/indoor-walk-test
GPU: True
grid (256, 448) | clip 40.0 - 170.0 dB | 9 channels


In [10]:
from pathlib import Path
DATASET = Path('SIM/dataset')
if not any(DATASET.glob('shard_*.npz')):
    from google.colab import drive
    drive.mount('/content/drive')
    root = Path('/content/drive/MyDrive')
    candidates = [root / 'indoor-walk-test-dataset',
                  root / 'SIM' / 'indoor-walk-test-dataset',
                  root / 'SIM' / 'dataset', root / 'dataset']
    candidates += [q for p in root.iterdir() if p.is_dir()
                   for q in [p] + [d for d in p.iterdir() if d.is_dir()]]
    DATASET = next((c for c in candidates if any(c.glob('shard_*.npz'))), None)
    assert DATASET, ('No shard_*.npz in the top two levels of My Drive. '
                     'Upload the local SIM/dataset folder (1.2 GB) first.')
print('dataset from:', DATASET)

SAVE_DIR = (DATASET.parent / 'checkpoints'
            if str(DATASET).startswith('/content/drive') else Path('.'))
SAVE_DIR.mkdir(exist_ok=True)
print('checkpoints ->', SAVE_DIR)

splits = json.load(open(DATASET / 'splits.json'))
shards = sorted(DATASET.glob('shard_*.npz'))
print(f'{len(shards)} shards, loading...')
tx_all, ff_all, tg_all, pid_all = [], [], [], []
for s in shards:
    d = np.load(s)
    tx_all.append(d['tx_pos']); ff_all.append(d['freq_feat'])
    tg_all.append(d['target']); pid_all.append(d['pos_id'])
TX = np.concatenate(tx_all); FF = np.concatenate(ff_all)
TG = np.concatenate(tg_all)
PID = np.concatenate(pid_all)
print(f'{len(TG)} samples, {TG.nbytes/1e9:.2f} GB in RAM')
idx = {k: np.nonzero(np.isin(PID, splits[k]))[0] for k in ('train','val','test')}
print({k: len(v) for k, v in idx.items()})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset from: /content/drive/MyDrive/SIM/indoor-walk-test-dataset
checkpoints -> /content/drive/MyDrive/SIM/checkpoints
20 shards, loading...
10000 samples, 2.29 GB in RAM
{'train': 8000, 'val': 1000, 'test': 1000}


In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

grid = np.load('SIM/grid_model.npy')
INSIDE = np.load('SIM/inside_mask.npy')
onehot = np.stack([(grid == c) for c in range(6)]).astype(np.float32)
yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)

def build_input(tx_x, tx_y, ff):
    """The 9-channel input. The SAME code must exist in JS (R6):
    onehot(6) + tx gaussian + freq const + log10(dist m)/DN."""
    x = np.empty((IN_CH, H, W), np.float32)
    x[:6] = onehot
    d = np.hypot(xx - tx_x, yy - tx_y)
    x[6] = np.exp(-d**2 / (2 * SIGMA**2))
    x[7] = ff
    x[8] = np.log10(np.maximum(d * CELL, 1.0)) / DN
    return x

class PLData(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]
        return (torch.from_numpy(build_input(TX[j,0], TX[j,1], FF[j])),
                torch.from_numpy(TG[j].astype(np.float32)))

def loaders(batch):
    return (DataLoader(PLData(idx['train']), batch_size=batch, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True),
            DataLoader(PLData(idx['val']), batch_size=batch, num_workers=2))

MASK = torch.from_numpy(INSIDE.astype(np.float32))   # loss counts indoor only

In [12]:
import torch.nn as nn

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))

class UNet(nn.Module):
    def __init__(self, base=64, in_ch=IN_CH):
        super().__init__()
        ch = [base, base*2, base*4, base*8]
        self.enc = nn.ModuleList(); prev = in_ch
        for c in ch: self.enc.append(block(prev, c)); prev = c
        self.pool = nn.MaxPool2d(2)
        self.mid = block(prev, prev*2)
        self.up, self.dec = nn.ModuleList(), nn.ModuleList()
        prev = prev*2
        for c in reversed(ch):
            self.up.append(nn.ConvTranspose2d(prev, c, 2, 2))
            self.dec.append(block(2*c, c)); prev = c
        self.head = nn.Conv2d(prev, 1, 1)          # R1
    def forward(self, x):
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x); x = self.pool(x)
        x = self.mid(x)
        for u, d in zip(self.up, self.dec):
            x = d(torch.cat([u(x), skips.pop()], 1))
        return self.head(x)[:, 0]

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'{sum(p.numel() for p in UNet().parameters())/1e6:.1f} M params')

31.0 M params


In [13]:
import time, math

GRAD_W = 0.1
RESUME_EVERY = 2   # write full training state to Drive every N epochs (~370 MB)

def grad_l1(a, b):
    return ((a[:,1:,:]-a[:,:-1,:]) - (b[:,1:,:]-b[:,:-1,:])).abs().mean() + \
           ((a[:,:,1:]-a[:,:,:-1]) - (b[:,:,1:]-b[:,:,:-1])).abs().mean()

def ckpt(seed):
    return SAVE_DIR / f'unet9_seed{seed}_best.pt'

def resume_file(seed):
    return SAVE_DIR / f'unet9_seed{seed}_resume.pt'

def train_one(seed, base=64, lr=1e-3, batch=16, epochs=150, patience=15):
    torch.manual_seed(seed); np.random.seed(seed)
    net = UNet(base).to(dev)
    opt = torch.optim.Adam(net.parameters(), lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=1e-5)
    scaler = torch.amp.GradScaler(dev)
    start_ep, best, best_ep = 0, 1e9, -1
    # TRUE RESUME: if a disconnect killed a previous run, pick up the epoch,
    # optimizer, LR schedule, and best score exactly where they were.
    if resume_file(seed).exists():
        st = torch.load(resume_file(seed), map_location=dev)
        net.load_state_dict(st['net']); opt.load_state_dict(st['opt'])
        sched.load_state_dict(st['sched']); scaler.load_state_dict(st['scaler'])
        start_ep, best, best_ep = st['epoch'] + 1, st['best'], st['best_ep']
        print(f'RESUMED seed {seed} at epoch {start_ep + 1}, best {best:.2f} dB')
    tr_dl, va_dl = loaders(batch)
    mask = MASK.to(dev); msum = mask.sum()
    for ep in range(start_ep, epochs):
        net.train(); t0 = time.time()
        for x, y in tr_dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.amp.autocast(dev):
                p = net(x)
                mse = (((p - y)**2) * mask).sum() / (msum * x.size(0))
                loss = mse + GRAD_W * grad_l1(p * mask, y * mask)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()
        net.eval(); se = n = 0
        with torch.no_grad():
            for x, y in va_dl:
                p = net(x.to(dev)).cpu()
                se += (((p - y)**2) * MASK).sum().item()
                n += MASK.sum().item() * y.size(0)
        rmse_db = PL_RNG * math.sqrt(se / n)
        if rmse_db < best:
            best, best_ep = rmse_db, ep
            torch.save(net.state_dict(), ckpt(seed))
        if ep % RESUME_EVERY == 0 or ep == epochs - 1:
            torch.save(dict(net=net.state_dict(), opt=opt.state_dict(),
                            sched=sched.state_dict(), scaler=scaler.state_dict(),
                            epoch=ep, best=best, best_ep=best_ep),
                       resume_file(seed))
        print(f'seed {seed} ep {ep+1:3d}  val RMSE {rmse_db:5.2f} dB (indoor)  '
              f'best {best:5.2f}  ({time.time()-t0:.0f}s)')
        if ep - best_ep >= patience:
            print(f'early stop (patience {patience})')
            resume_file(seed).unlink(missing_ok=True)   # finished: clear resume
            break
    return best

In [ ]:
# R9: seeds {0,1,2}. QUICK=True = one seed for the first pass.
QUICK = True
SEEDS = [0] if QUICK else [0, 1, 2]
results = {s: train_one(s) for s in SEEDS}
vals = list(results.values())
print(f'val RMSE over {len(SEEDS)} seed(s): '
      f'{np.mean(vals):.2f} ± {np.std(vals):.2f} dB  (target <= 3 dB)')

seed 0 ep   1  val RMSE 11.53 dB (indoor)  best 11.53  (66s)
seed 0 ep   2  val RMSE 10.81 dB (indoor)  best 10.81  (65s)
seed 0 ep   3  val RMSE  9.22 dB (indoor)  best  9.22  (66s)
seed 0 ep   4  val RMSE  9.44 dB (indoor)  best  9.22  (65s)


## Baselines (spec C.4) — validation for selection; test only in the final section (R5).

In [ ]:
import importlib.util
spec_pa = importlib.util.spec_from_file_location('pa', 'SIM/phase_a.py')
pa = importlib.util.module_from_spec(spec_pa); spec_pa.loader.exec_module(pa)
F_LO, F_HI = np.log10(NORM['freq_log_lo_mhz']), np.log10(NORM['freq_log_hi_mhz'])
dist_m = lambda j: np.maximum(np.hypot(xx - TX[j,0], yy - TX[j,1]) * CELL, 1.0)
f_of = lambda j: 10 ** (FF[j] * (F_HI - F_LO) + F_LO)
ins = INSIDE

def fspl_map(j):
    return 32.44 + 20*np.log10(f_of(j)) - 60 + 20*np.log10(dist_m(j))

sub = np.random.default_rng(0).choice(idx['train'], 400, replace=False)
num = den = 0.0
for j in sub:
    pl = TG[j].astype(np.float32) * PL_RNG + PL_LO
    ex = pl - (32.44 + 20*np.log10(f_of(j)) - 60)
    ld = 10*np.log10(dist_m(j))
    num += (ex[ins] * ld[ins]).sum(); den += (ld[ins]**2).sum()
N_FIT = num / den
print(f'fitted log-distance n = {N_FIT:.2f}')

def logdist_map(j):
    return 32.44 + 20*np.log10(f_of(j)) - 60 + 10*N_FIT*np.log10(dist_m(j))

def inh38901_map(j, los):
    d = dist_m(j); fghz = f_of(j) / 1000
    pl_los = 32.4 + 17.3*np.log10(d) + 20*np.log10(fghz)
    pl_nlos = np.maximum(pl_los, 17.3 + 38.3*np.log10(d) + 24.9*np.log10(fghz))
    return np.where(los, pl_los, pl_nlos)

MODEL_GRID = np.load('SIM/grid_model.npy')

def eval_maps(ids, model=None, kind='unet', los_cache=None):
    se = ae = be = n = 0
    for j in ids:
        gt = TG[j].astype(np.float32) * PL_RNG + PL_LO
        if kind == 'unet':
            xin = torch.from_numpy(build_input(TX[j,0], TX[j,1], FF[j]))[None]
            with torch.no_grad():
                p = model(xin.to(dev))[0].cpu().numpy()
            pr = np.clip(p, 0, 1) * PL_RNG + PL_LO
        elif kind == 'fspl':    pr = np.clip(fspl_map(j), PL_LO, PL_LO+PL_RNG)
        elif kind == 'logdist': pr = np.clip(logdist_map(j), PL_LO, PL_LO+PL_RNG)
        elif kind == '38901':
            key = tuple(TX[j])
            if key not in los_cache:
                _, k = pa.pathloss_map(MODEL_GRID,
                                       (float(TX[j,0]), float(TX[j,1])),
                                       3500.0, CELL, return_crossings=True)
                los_cache[key] = k == 0
            pr = np.clip(inh38901_map(j, los_cache[key]), PL_LO, PL_LO+PL_RNG)
        e = (pr - gt)[ins]
        se += (e**2).sum(); ae += np.abs(e).sum(); be += e.sum(); n += e.size
    return dict(rmse=np.sqrt(se/n), mae=ae/n, bias=be/n)

net = UNet().to(dev)
net.load_state_dict(torch.load(ckpt(SEEDS[0]), map_location=dev))
net.eval()
val_sub = np.random.default_rng(1).choice(idx['val'], 200, replace=False)
los_cache = {}
for kind in ('fspl', 'logdist', '38901', 'unet'):
    r = eval_maps(val_sub, net, kind, los_cache)
    print(f"VAL {kind:8s} RMSE {r['rmse']:6.2f}  MAE {r['mae']:6.2f}  bias {r['bias']:+6.2f} dB")

## Final test evaluation — R5: run ONCE, after all selection is done.

In [ ]:
test_sub = idx['test']
for kind in ('fspl', 'logdist', '38901', 'unet'):
    r = eval_maps(test_sub, net, kind, los_cache)
    print(f"TEST {kind:8s} RMSE {r['rmse']:6.2f}  MAE {r['mae']:6.2f}  bias {r['bias']:+6.2f} dB")

In [ ]:
import matplotlib.pyplot as plt
js = idx['test'][:3]
fig, axes = plt.subplots(3, 3, figsize=(16, 7.5), squeeze=False)
for r, j in enumerate(js):
    gt = TG[j].astype(np.float32) * PL_RNG + PL_LO
    xin = torch.from_numpy(build_input(TX[j,0], TX[j,1], FF[j]))[None]
    with torch.no_grad():
        p = net(xin.to(dev))[0].cpu().numpy()
    pr = np.clip(p, 0, 1) * PL_RNG + PL_LO
    for c, (img, ttl, cm, lo, hi) in enumerate([
            (gt, 'simulated PL (dB)', 'turbo', 40, 170),
            (pr, 'UNet prediction', 'turbo', 40, 170),
            (np.abs(pr-gt), '|error| (dB)', 'magma', 0, 10)]):
        a = axes[r][c]
        im = a.imshow(np.where(ins, img, np.nan), cmap=cm, vmin=lo, vmax=hi)
        a.set_title(ttl if r == 0 else '', fontsize=10); a.axis('off')
        fig.colorbar(im, ax=a, shrink=0.7)
plt.tight_layout()

In [ ]:
# D.3 physical sanity checks
j = idx['test'][0]
xin = torch.from_numpy(build_input(TX[j,0], TX[j,1], FF[j]))[None]
with torch.no_grad():
    p = net(xin.to(dev))[0].cpu().numpy() * PL_RNG + PL_LO
fx, fy = TX[j]
ray = p[int(fy), int(fx):int(fx)+40]
print('monotone LOS decay (fraction of increasing steps):',
      (np.diff(ray) > -0.5).mean().round(2))
print('cells below FSPL:', f'{100*(p < fspl_map(j) - 1)[ins].mean():.2f}% (want ~0)')
x24 = build_input(TX[j,0], TX[j,1], 0.0)
with torch.no_grad():
    p24 = net(torch.from_numpy(x24)[None].to(dev))[0].cpu().numpy() * PL_RNG + PL_LO
print('median(PL@2.4GHz <= PL@current):', f'{np.median((p24 <= p + 0.5)[ins]):.0f} (want 1)')

In [ ]:
# F.1/F.2: export + parity gate (<= 0.1 dB over 50 test maps)
net.cpu().eval()
torch.onnx.export(net, torch.zeros(1, IN_CH, H, W), 'pl_unet.onnx',
                  input_names=['x'], output_names=['pl_norm'], opset_version=17)
import onnxruntime as ort_rt
sess = ort_rt.InferenceSession('pl_unet.onnx')
worst = 0
for j in idx['test'][:50]:
    xin = build_input(TX[j,0], TX[j,1], FF[j])
    with torch.no_grad():
        pt = net(torch.from_numpy(xin)[None])[0].numpy()
    ox = sess.run(None, {'x': xin[None]})[0][0]
    worst = max(worst, np.abs(pt - ox).max() * PL_RNG)
print(f'ONNX parity: max |diff| {worst:.4f} dB (must be <= 0.1)')
assert worst <= 0.1, 'PARITY FAIL - do not ship this file'
net.to(dev)
import shutil
if str(SAVE_DIR).startswith('/content/drive'):
    shutil.copy('pl_unet.onnx', SAVE_DIR / 'pl_unet.onnx')
    print('copied to', SAVE_DIR / 'pl_unet.onnx')
from google.colab import files
files.download('pl_unet.onnx')   # -> SIM/web/pl_unet.onnx on your Mac